# LinkedIn distribution tracker — Visualizing the Unseen

Started 2026-08-15 after days 5-7 impressions collapsed (111, 46, 27) following a flat baseline on days 1-4 (207, 200, 210, 195), with near-nil engagement (1 comment + 2 likes on the best days) the entire time. Working hypothesis: LinkedIn's account-level distribution throttle from sustained low engagement + daily cadence — not an "AI slop" content flag (that would suppress reach per-post, not compound across posts, and Simon's own engagement on *other* people's posts lands fine).

**Open confound this notebook exists to test:** raw impression totals aren't comparable across posts checked at different ages — day 1 had 6 days to accumulate by the first check, day 7 had ~0. `imp_per_day` (impressions / days since posted) is the metric that actually separates "throttled" from "just hasn't finished its normal decay curve yet."

Seed `post_date` for days 4-7 is **inferred** from file mtimes (daily cadence 08-09 through 08-15) — not confirmed. Fix in the CSV if wrong.

Cadence changed 2026-08-15: daily -> every 3-5 days. Log a reading here each day regardless of whether a new post went up.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CSV_PATH = Path("data/linkedin_impressions.csv")
TODAY = "2026-08-15"

# post_day, post_date, impressions, likes, comments
# post_date for days 4-7 INFERRED from daily cadence -- confirm against real post times
SEED = [
    (1, "2026-08-09", 207, np.nan, np.nan),
    (2, "2026-08-10", 200, 2, 1),
    (3, "2026-08-11", 210, np.nan, np.nan),
    (4, "2026-08-12", 195, 2, 1),
    (5, "2026-08-13", 111, np.nan, np.nan),
    (6, "2026-08-14", 46, np.nan, np.nan),
    (7, "2026-08-15", 27, np.nan, np.nan),
]

if not CSV_PATH.exists():
    CSV_PATH.parent.mkdir(exist_ok=True)
    seed_df = pd.DataFrame(SEED, columns=["post_day", "post_date", "impressions", "likes", "comments"])
    seed_df["check_date"] = TODAY
    seed_df.to_csv(CSV_PATH, index=False)

df = pd.read_csv(CSV_PATH, parse_dates=["post_date", "check_date"])
df

In [ ]:
def log_check(post_day, impressions, likes=np.nan, comments=np.nan, check_date=None, post_date=None):
    """Append a daily reading. post_date only required the first time a post_day is logged."""
    d = pd.read_csv(CSV_PATH, parse_dates=["post_date", "check_date"])
    if post_date is None:
        existing = d.loc[d.post_day == post_day, "post_date"]
        if existing.empty:
            raise ValueError(f"post_day {post_day} has no post_date on file -- pass one")
        post_date = existing.iloc[0]
    row = {
        "post_day": post_day,
        "post_date": post_date,
        "check_date": check_date or pd.Timestamp.today().normalize(),
        "impressions": impressions,
        "likes": likes,
        "comments": comments,
    }
    d = pd.concat([d, pd.DataFrame([row])], ignore_index=True)
    d.to_csv(CSV_PATH, index=False)
    return d

# example: log_check(7, impressions=40, likes=1, comments=0)
# example for a brand-new post: log_check(8, impressions=50, post_date="2026-08-19")

In [ ]:
df = pd.read_csv(CSV_PATH, parse_dates=["post_date", "check_date"])
df["days_since_posted"] = (df["check_date"] - df["post_date"]).dt.total_seconds() / 86400
df["imp_per_day"] = df["impressions"] / df["days_since_posted"].replace(0, np.nan)

# most recent reading per post
latest = df.sort_values("check_date").groupby("post_day").tail(1).sort_values("post_day")
latest[["post_day", "post_date", "days_since_posted", "impressions", "imp_per_day", "likes", "comments"]]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(latest["post_day"], latest["impressions"], marker="o", color="#2f6fed")
axes[0].set_title("Raw impressions per post (NOT age-adjusted)")
axes[0].set_xlabel("Post day #")
axes[0].set_ylabel("Impressions")

axes[1].plot(latest["post_day"], latest["imp_per_day"], marker="o", color="#e0552f")
axes[1].set_title("Impressions / day since posting (age-adjusted)")
axes[1].set_xlabel("Post day #")
axes[1].set_ylabel("Impressions per day")
axes[1].axvline(7.5, color="gray", linestyle="--", linewidth=1)

plt.tight_layout()
plt.savefig("linkedin_impressions_trend.png", dpi=120)
plt.show()

In [ ]:
def _demo():
    tmp = pd.DataFrame([
        {"post_day": 1, "post_date": pd.Timestamp("2026-08-09"), "check_date": pd.Timestamp("2026-08-11"), "impressions": 100},
        {"post_day": 1, "post_date": pd.Timestamp("2026-08-09"), "check_date": pd.Timestamp("2026-08-13"), "impressions": 200},
    ])
    tmp["days_since_posted"] = (tmp["check_date"] - tmp["post_date"]).dt.total_seconds() / 86400
    tmp["imp_per_day"] = tmp["impressions"] / tmp["days_since_posted"]
    assert abs(tmp["imp_per_day"].iloc[0] - 50) < 1e-9
    assert abs(tmp["imp_per_day"].iloc[1] - 50) < 1e-9
    print("self-check ok")

_demo()